## 

## Extracting photos from movies and producing dataset

In [9]:
import cv2
import os

VIDEO_ROOT = "F:/python/Deep/project2/dataset/videos"
IMAGE_ROOT = "F:/python/Deep/project2/dataset/images"

FRAME_STEP = 8

classes = ["one", "two", "three", "four", "five", "ok"]

for cls in classes:
    video_dir = os.path.join(VIDEO_ROOT, cls)
    image_dir = os.path.join(IMAGE_ROOT, cls)
    os.makedirs(image_dir, exist_ok=True)

    for video_name in os.listdir(video_dir):
        if not video_name.endswith((".mp4", ".avi", ".mov")):
            continue

        cap = cv2.VideoCapture(os.path.join(video_dir, video_name))
        frame_id = 0
        save_id = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_id % FRAME_STEP == 0:
                img_name = f"{cls}_{video_name[:-4]}_{save_id:05d}.jpg"
                cv2.imwrite(os.path.join(image_dir, img_name), frame)
                save_id += 1

            frame_id += 1

        cap.release()

print("Frame extraction finished.")

Frame extraction finished.


## Labeling

In [30]:
import cv2
import mediapipe as mp
import os
import math

IMAGE_ROOT = "F:/python/Deep/project2/dataset/images"
LABEL_ROOT = "F:/python/Deep/project2/dataset/labels"

CLASSES = ["one", "two", "three", "four", "five", "ok"]
CLASS_ID = {c:i for i,c in enumerate(CLASSES)}

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.6
)

PADDING = 0.2

for cls in CLASSES:
    img_dir = os.path.join(IMAGE_ROOT, cls)
    lbl_dir = os.path.join(LABEL_ROOT, cls)
    os.makedirs(lbl_dir, exist_ok=True)

    for img_name in os.listdir(img_dir):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w, _ = img.shape
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        res = hands.process(rgb)

        if not res.multi_hand_landmarks:
            continue

        lm = res.multi_hand_landmarks[0].landmark

        xs = [p.x for p in lm]
        ys = [p.y for p in lm]

        x1, x2 = min(xs), max(xs)
        y1, y2 = min(ys), max(ys)

        bw = x2 - x1
        bh = y2 - y1

        x1 = max(0, x1 - bw * PADDING)
        y1 = max(0, y1 - bh * PADDING)
        x2 = min(1, x2 + bw * PADDING)
        y2 = min(1, y2 + bh * PADDING)

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bw = x2 - x1
        bh = y2 - y1

        label_path = os.path.join(lbl_dir, img_name.replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            f.write(f"{CLASS_ID[cls]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

print("Labeling finished successfully.")

Labeling finished successfully.


## Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  

model.train(
    data="F:\python\Deep\project2\dataset\yolo\data.yaml",
    epochs=40,
    imgsz=640,
    batch=16,         
    device="cpu",
    workers=0       
)

Ultralytics 8.4.14  Python-3.10.0 torch-2.10.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\python\Deep\project2\dataset\yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D950BD32B0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
 

## Testing with webcam

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

model.predict(
    source=0,      # webcam
    show=True,
    conf=0.5,
    device="cpu"
)


1/1: 0... Success  (inf frames of shape 640x480 at 30.00 FPS)

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

0: 480x640 (no detections), 108.7ms
0: 480x640 (no detections), 92.3ms
0: 480x640 (no detections), 70.6ms
0: 480x640 (no detections), 66.8ms
0: 480x640 (no detections), 64.9ms
0: 480x640 (no detections), 65.4ms
0: 480x640 (no detections), 69.9ms
0: 480x640 (no detections), 82.1ms
0: 480x640 (no detections), 64.4ms
0: 480x640 (no detections), 67.5ms
0: 480x640 (no detections), 73.9ms
0: 480x640 (

KeyboardInterrupt: 